# WAFER-GUARD · Fab-7 Excursion Detector — **repaired** pipeline

This is the audited and corrected version of the original notebook (`bnypi.ipynb`).
Every change is tagged in the code with `# FIX[n]` and explained in `AUDIT.md`.

| # | Silent defect in the original | Repair in this notebook |
|---|---|---|
| 1 | **Data leakage.** The imputer and scaler were fit on all rows, SMOTE ran *before* the split, and the split was random, so the future and the test set leaked into training. The test set was also 50 % synthetic and balanced. | Chronological train / val / test split. All statistics are fit on **train only**. No SMOTE: class imbalance is handled with `pos_weight`. The test set keeps its natural prevalence. |
| 2 | **Corrupted loss gradients.** `reconstructed` is `[B,F]` and `batch_x` is `[B,1,F]`, so they broadcast silently to `[B,B,F]` and the decoder learns the batch mean. | Tensors are kept at `[B,F]`. A loss function asserts the shapes and masks missing or placeholder cells. |
| 3 | **Invariant blindness.** The legacy PS-L sentinel (`sensor_0 == 0.0`) and the firmware codes in `sensor_1` were scaled and imputed as continuous values. | A rule-based legacy flag and one-hot error codes. The sentinel and code slots are excluded from scaling, imputation statistics and the reconstruction loss. |

The decision threshold is selected on the validation window only. The model is scored **once** on the untouched test window.

## 0 · Environment

In [1]:
# Dependencies: see requirements.txt  ->  pip install -r requirements.txt
import os, json, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             average_precision_score, confusion_matrix, precision_recall_curve)
import joblib

warnings.filterwarnings("ignore", category=RuntimeWarning)   # nan-reductions on all-NaN columns

SEED = 1337
def seed_all(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
seed_all(SEED)
torch.use_deterministic_algorithms(True)
DEVICE = torch.device("cpu")          # tiny tabular problem; CPU keeps runs bit-reproducible

HP = dict(batch_size=64, max_epochs=60, patience=10, lr=1e-3, weight_decay=1e-4,
          hidden=256, latent=32, lambda_clf=1.0,
          split=(0.60, 0.20, 0.20),          # chronological train / val / test
          max_missing=0.50,                  # drop channels missing in >50 % of TRAIN rows
          indicator_min_missing=0.01,        # add a missing-flag for channels missing >=1 % in TRAIN
          z_clip=10.0)                       # robust to drifted/outlier telemetry at inference
NATURAL_PREVALENCE = 0.0664                  # full-dataset defect rate (104 / 1567)
print(f"torch {torch.__version__} | device={DEVICE}")

torch 2.12.1+cpu | device=cpu


## 1 · Data
Rows are sorted by `timestamp`. The fab drifts over time, so time order matters for every later step.

In [2]:
df = pd.read_csv("secom_telemetry_modified.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)
assert df["timestamp"].is_monotonic_increasing

SENSOR_COLS = [c for c in df.columns if c.startswith("sensor_")]
y_all = (df["label"].to_numpy() == 1).astype(np.int64)            # 1 = defective

print(f"rows {len(df)} | sensors {len(SENSOR_COLS)} | defect rate {y_all.mean():.2%} "
      f"({y_all.sum()} fail / {(1 - y_all).sum()} pass)")
print(f"span {df.timestamp.min()} -> {df.timestamp.max()}")
print("defect rate per chronological decile:",
      [f"{c.mean():.1%}" for c in np.array_split(y_all, 10)])

rows 1567 | sensors 590 | defect rate 6.64% (104 fail / 1463 pass)
span 2008-07-19 11:55:00 -> 2008-10-17 06:07:00
defect rate per chronological decile: ['12.7%', '15.3%', '8.9%', '4.5%', '1.3%', '5.7%', '1.3%', '6.4%', '8.3%', '1.9%']


## 2 · Chronological split  `# FIX[1]`
Oldest 60 % → **train**, next 20 % → **validation** (early stopping and threshold), newest 20 % → **test**.
The test window is never resampled. Its defect rate is whatever the fab actually produced in that period.

In [3]:
n = len(df)
i1 = int(HP["split"][0] * n); i2 = int((HP["split"][0] + HP["split"][1]) * n)
idx_tr, idx_va, idx_te = np.arange(0, i1), np.arange(i1, i2), np.arange(i2, n)

# causality: every training timestamp strictly precedes every val timestamp, and so on
assert df.timestamp[idx_tr].max() < df.timestamp[idx_va].min()
assert df.timestamp[idx_va].max() < df.timestamp[idx_te].min()

for name, ix in [("train", idx_tr), ("val", idx_va), ("test", idx_te)]:
    print(f"{name:5s} n={len(ix):4d}  defects={y_all[ix].sum():3d} ({y_all[ix].mean():.2%})  "
          f"{df.timestamp[ix].min()} -> {df.timestamp[ix].max()}")

train n= 940  defects= 76 (8.09%)  2008-07-19 11:55:00 -> 2008-09-20 05:34:00
val   n= 313  defects= 11 (3.51%)  2008-09-20 06:08:00 -> 2008-10-02 19:25:00
test  n= 314  defects= 17 (5.41%)  2008-10-02 20:54:00 -> 2008-10-17 06:07:00


## 3 · Legacy Probe Station PS-L tool-matching  `# FIX[3]`
* `sensor_0 == 0.0` is a **sentinel** meaning "routed through PS-L". It is not a measurement.
* On PS-L wafers `sensor_1` is a **categorical firmware error code** (100 / 200 / 300), not a reading.

These rules apply **row by row** and have no fitted state, so they cannot leak information across time.
A code we have never seen is routed to `s1_code_unknown`. It is not silently treated as a number.

In [4]:
LEGACY_SENTINEL = 0.0
LEGACY_CODES = (100.0, 200.0, 300.0)

def legacy_view(frame: pd.DataFrame):
    # Returns (is_legacy[N], code_onehot[N,4], na_mask[N,F]) for one frame. Stateless, row-wise.
    s0 = frame["sensor_0"].to_numpy(np.float64)
    s1 = frame["sensor_1"].to_numpy(np.float64)
    is_legacy = (s0 == LEGACY_SENTINEL)
    known = np.stack([s1 == c for c in LEGACY_CODES], axis=1)
    # a legacy row whose sensor_1 is not a known code -> 'unknown code' bucket
    unknown = is_legacy & ~known.any(1)
    # defensive: a known code value on a non-legacy row is still a code, never a reading
    code_slot = is_legacy | known.any(1)
    onehot = np.concatenate([known & code_slot[:, None], unknown[:, None]], 1).astype(np.float32)
    na = np.zeros((len(frame), len(SENSOR_COLS)), dtype=bool)   # 'not applicable' slots
    na[is_legacy, SENSOR_COLS.index("sensor_0")] = True
    na[code_slot, SENSOR_COLS.index("sensor_1")] = True
    return is_legacy, onehot, na

is_leg, codes, _ = legacy_view(df)
print(f"legacy PS-L wafers: {is_leg.sum()} ({is_leg.mean():.1%})")
print("code counts (100/200/300/unknown):", codes.sum(0).astype(int).tolist())
print("defect rate legacy vs current-gen:",
      f"{y_all[is_leg].mean():.2%} vs {y_all[~is_leg].mean():.2%}")

legacy PS-L wafers: 146 (9.3%)
code counts (100/200/300/unknown): [61, 35, 50, 0]
defect rate legacy vs current-gen: 0.00% vs 7.32%


## 4 · Causal preprocessor  `# FIX[1]` `# FIX[3]`
`fit()` sees **training rows only**. `transform()` is then applied unchanged to validation, test and any future production wafer.

* Legacy sentinel and code slots become *not-applicable*. They are excluded from every mean, std and median.
* Channels that are constant, or missing in more than 50 % of rows, **in train** are dropped.
* Remaining channels are z-scored with train statistics and clipped to ±10σ.
* Genuinely missing cells get the train median plus a missing-indicator, for channels with ≥1 % train missingness.
* Not-applicable legacy slots get a fixed 0 placeholder. It is never used as a target (`recon_mask = 0`). The model learns its meaning from `is_legacy` and the one-hot codes.

In [5]:
class CausalPreprocessor:
    def fit(self, frame: pd.DataFrame):
        X = frame[SENSOR_COLS].to_numpy(np.float64).copy()
        _, _, na = legacy_view(frame)
        X[na] = np.nan                                   # sentinel / codes never enter statistics
        miss = np.isnan(X).mean(0)
        std = np.nanstd(X, 0)
        self.keep_ = np.where((miss <= HP["max_missing"]) & (np.nan_to_num(std) > 0))[0]
        Xk = X[:, self.keep_]
        self.mean_, self.std_ = np.nanmean(Xk, 0), np.nanstd(Xk, 0)
        self.median_z_ = (np.nanmedian(Xk, 0) - self.mean_) / self.std_
        genuine_miss = (np.isnan(frame[SENSOR_COLS].to_numpy(np.float64)) & ~na)[:, self.keep_].mean(0)
        self.ind_ = np.where(genuine_miss >= HP["indicator_min_missing"])[0]
        self.feature_names_ = ([SENSOR_COLS[k] for k in self.keep_]
                               + [f"miss_{SENSOR_COLS[self.keep_[j]]}" for j in self.ind_]
                               + ["is_legacy", "s1_code_100", "s1_code_200", "s1_code_300", "s1_code_unknown"])
        self.n_cont_ = len(self.keep_)
        return self

    def transform(self, frame: pd.DataFrame):
        raw = frame[SENSOR_COLS].to_numpy(np.float64)
        is_legacy, onehot, na = legacy_view(frame)
        Xk = raw[:, self.keep_]
        na_k = na[:, self.keep_]
        miss_k = np.isnan(Xk) & ~na_k                    # genuinely missing readings
        Z = np.clip((Xk - self.mean_) / self.std_, -HP["z_clip"], HP["z_clip"])
        Z = np.where(miss_k, self.median_z_, Z)          # train-median imputation
        Z = np.where(na_k, 0.0, Z)                       # placeholder for not-applicable slots
        feats = np.concatenate([Z, miss_k[:, self.ind_].astype(np.float64),
                                is_legacy[:, None].astype(np.float64), onehot], 1).astype(np.float32)
        # reconstruction targets: only genuinely observed readings + the categorical / indicator block
        mask = np.ones_like(feats, dtype=np.float32)
        mask[:, :self.n_cont_] = (~miss_k & ~na_k).astype(np.float32)
        assert np.isfinite(feats).all()
        return feats, mask

prep = CausalPreprocessor().fit(df.iloc[idx_tr])
(X_tr, M_tr), (X_va, M_va), (X_te, M_te) = (prep.transform(df.iloc[ix]) for ix in (idx_tr, idx_va, idx_te))
y_tr, y_va, y_te = y_all[idx_tr], y_all[idx_va], y_all[idx_te]
F = X_tr.shape[1]
print(f"features: {F} = {prep.n_cont_} continuous + {len(prep.ind_)} missing-flags + 5 legacy/categorical")
j1 = list(prep.keep_).index(SENSOR_COLS.index("sensor_1"))
print(f"sensor_1 train stats used for scaling: mean={prep.mean_[j1]:.2f} std={prep.std_[j1]:.2f} "
      f"(genuine readings only; codes excluded)")
print(f"X_tr {X_tr.shape} | X_va {X_va.shape} | X_te {X_te.shape}")

features: 487 = 444 continuous + 38 missing-flags + 5 legacy/categorical
sensor_1 train stats used for scaling: mean=2494.87 std=85.66 (genuine readings only; codes excluded)
X_tr (940, 487) | X_va (313, 487) | X_te (314, 487)


## 5 · Tensors & loaders  `# FIX[2]`
Features stay **`[N, F]`**. The original added a channel axis (`unsqueeze(1)`), which caused the broadcasting bug.

In [6]:
def make_loader(X, M, y, shuffle, seed=SEED):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(M), torch.from_numpy(y).float())
    g = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size=HP["batch_size"], shuffle=shuffle,
                      generator=g if shuffle else None, drop_last=False)

xb, mb, yb = next(iter(make_loader(X_tr, M_tr, y_tr, True)))
print("batch shapes  x", tuple(xb.shape), "| mask", tuple(mb.shape), "| y", tuple(yb.shape))

batch shapes  x (64, 487) | mask (64, 487) | y (64,)


## 6 · Model — supervised autoencoder (same architecture as the original; strict input rank)

In [7]:
class WaferSAE(nn.Module):
    def __init__(self, d_in: int, d_hid: int, d_lat: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hid), nn.LayerNorm(d_hid), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(d_hid, d_hid // 2), nn.LayerNorm(d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_lat),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_lat, d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_hid), nn.GELU(),
            nn.Linear(d_hid, d_in),
        )
        self.head = nn.Linear(d_lat, 1)

    def forward(self, x):
        # FIX[2]: no silent flatten; reject anything that is not [B, F]
        assert x.dim() == 2, f"expected [B, F], got {tuple(x.shape)}"
        z = self.encoder(x)
        return self.decoder(z), self.head(z).squeeze(-1), z

## 7 · Shape-checked loss  `# FIX[2]`
* Reconstruction: masked MSE over **observed cells only**, using an element-wise `[B,F]` comparison.
* Classification: `BCEWithLogits` on `[B]` logits against `[B]` labels. `pos_weight` = train negatives / positives replaces SMOTE.

In [8]:
def sae_loss(recon, x, mask, logits, y, pos_weight, lam):
    B = x.shape[0]
    assert recon.shape == x.shape == mask.shape and x.dim() == 2, (recon.shape, x.shape, mask.shape)
    assert logits.shape == y.shape == (B,), (logits.shape, y.shape)
    recon_loss = ((recon - x) ** 2 * mask).sum() / mask.sum().clamp(min=1.0)
    clf_loss = Fnn.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
    return recon_loss + lam * clf_loss, recon_loss, clf_loss

# regression test: the original [B,1,F] input must now fail loudly instead of broadcasting
_m = WaferSAE(F, 16, 4)
try:
    _m(torch.zeros(4, 1, F)); raise RuntimeError("rank check did not fire")
except AssertionError as e:
    print("OK - rank-3 input rejected:", e)
try:
    sae_loss(torch.zeros(4, F), torch.zeros(4, 1, F), torch.ones(4, 1, F), torch.zeros(4), torch.zeros(4),
             torch.tensor(1.0), 1.0); raise RuntimeError("shape check did not fire")
except AssertionError:
    print("OK - mismatched loss shapes rejected")

OK - rank-3 input rejected: expected [B, F], got (4, 1, 487)
OK - mismatched loss shapes rejected


## 8 · Training (early stopping on validation average precision)

In [9]:
@torch.no_grad()
def predict_proba(model, X):
    model.eval()
    return torch.sigmoid(model(torch.from_numpy(X).to(DEVICE))[1]).cpu().numpy()

def train_sae(X_tr, M_tr, y_tr, X_va, y_va, seed=SEED, verbose=True):
    seed_all(seed)
    model = WaferSAE(X_tr.shape[1], HP["hidden"], HP["latent"]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HP["max_epochs"])
    pos_weight = torch.tensor((1 - y_tr).sum() / max(y_tr.sum(), 1), dtype=torch.float32)
    loader = make_loader(X_tr, M_tr, y_tr, shuffle=True, seed=seed)
    best = dict(ap=-1.0, epoch=0, state=None); hist = []
    for epoch in range(1, HP["max_epochs"] + 1):
        model.train(); tot = rec = clf = 0.0
        for xb, mb, yb in loader:
            r, logit, _ = model(xb.to(DEVICE))
            loss, lr_, lc_ = sae_loss(r, xb, mb, logit, yb, pos_weight, HP["lambda_clf"])
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0); opt.step()
            tot += loss.item() * len(xb); rec += lr_.item() * len(xb); clf += lc_.item() * len(xb)
        sched.step()
        va_ap = average_precision_score(y_va, predict_proba(model, X_va))
        hist.append(dict(epoch=epoch, total=tot / len(X_tr), recon=rec / len(X_tr), clf=clf / len(X_tr), val_ap=va_ap))
        if va_ap > best["ap"]:
            best = dict(ap=va_ap, epoch=epoch, state={k: v.clone() for k, v in model.state_dict().items()})
        if verbose and (epoch == 1 or epoch % 5 == 0):
            h = hist[-1]
            print(f"epoch {epoch:02d} | total {h['total']:.4f} | recon {h['recon']:.4f} | "
                  f"clf {h['clf']:.4f} | val AP {va_ap:.4f}")
        if epoch - best["epoch"] >= HP["patience"]:
            break
    model.load_state_dict(best["state"])
    if verbose:
        print(f"best epoch {best['epoch']} (val AP {best['ap']:.4f}); pos_weight={pos_weight.item():.2f}")
    return model, pd.DataFrame(hist)

model, history = train_sae(X_tr, M_tr, y_tr, X_va, y_va)

epoch 01 | total 2.0312 | recon 0.7943 | clf 1.2369 | val AP 0.0363


epoch 05 | total 0.7493 | recon 0.7087 | clf 0.0406 | val AP 0.0376


epoch 10 | total 0.6059 | recon 0.6037 | clf 0.0022 | val AP 0.0337


best epoch 3 (val AP 0.0398); pos_weight=11.37


## 9 · Decision threshold (validation window only)
The original used a fixed 0.5 threshold, which is meaningless after training on a 50/50 synthetic distribution.
Here the threshold that maximises F1 on the **validation** window is frozen before the test set is touched.

In [10]:
def pick_threshold(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
    return float(thr[np.argmax(f1)])

p_va = predict_proba(model, X_va)
THRESHOLD = pick_threshold(y_va, p_va)
print(f"validation: AP {average_precision_score(y_va, p_va):.4f} | ROC-AUC {roc_auc_score(y_va, p_va):.4f} "
      f"| F1@thr {f1_score(y_va, p_va >= THRESHOLD):.4f} | threshold {THRESHOLD:.4f}")

validation: AP 0.0398 | ROC-AUC 0.5536 | F1@thr 0.0933 | threshold 0.0194


## 10 · Final evaluation — untouched test window at natural prevalence
* **Primary metric: PR-AUC (average precision).** At ~5–7 % prevalence ROC-AUC flatters a model. A random scorer's AP equals the prevalence, so the reference point is easy to state.
* F1 / precision / recall are reported at the frozen validation threshold, with 95 % bootstrap CIs (2 000 resamples).
* The test window's measured prevalence differs from the dataset-wide 6.64 % because the fab drifts. So precision and F1 are **also** re-expressed at π = 6.64 %, using the measured TPR/FPR (Bayes' rule, no resampling).

In [11]:
def evaluate(y, p, thr, n_boot=2000, seed=SEED):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    tpr, fpr = tp / max(tp + fn, 1), fp / max(fp + tn, 1)
    pi = NATURAL_PREVALENCE
    prec_pi = tpr * pi / max(tpr * pi + fpr * (1 - pi), 1e-12)
    out = dict(n=len(y), defects=int(y.sum()), prevalence=y.mean(),
               ap=average_precision_score(y, p), roc_auc=roc_auc_score(y, p),
               f1=f1_score(y, pred, zero_division=0), precision=precision_score(y, pred, zero_division=0),
               recall=recall_score(y, pred), tp=tp, fp=fp, fn=fn, tn=tn,
               precision_at_6_64=prec_pi,
               f1_at_6_64=2 * prec_pi * tpr / max(prec_pi + tpr, 1e-12))
    rng = np.random.default_rng(seed); boots = []
    for _ in range(n_boot):
        b = rng.integers(0, len(y), len(y))
        if y[b].sum() in (0, len(b)):
            continue
        boots.append((average_precision_score(y[b], p[b]), roc_auc_score(y[b], p[b]),
                      f1_score(y[b], p[b] >= thr, zero_division=0)))
    lo, hi = np.percentile(np.array(boots), [2.5, 97.5], axis=0)
    out.update(ap_ci=(lo[0], hi[0]), roc_auc_ci=(lo[1], hi[1]), f1_ci=(lo[2], hi[2]))
    return out

def show(r, title):
    print(f"--- {title} ---")
    print(f"n={r['n']}  defects={r['defects']}  prevalence={r['prevalence']:.2%}")
    print(f"PR-AUC (AP) {r['ap']:.4f}  95% CI [{r['ap_ci'][0]:.3f}, {r['ap_ci'][1]:.3f}]   "
          f"(random = {r['prevalence']:.4f})")
    print(f"ROC-AUC     {r['roc_auc']:.4f}  95% CI [{r['roc_auc_ci'][0]:.3f}, {r['roc_auc_ci'][1]:.3f}]")
    print(f"F1          {r['f1']:.4f}  95% CI [{r['f1_ci'][0]:.3f}, {r['f1_ci'][1]:.3f}]   "
          f"precision {r['precision']:.4f}  recall {r['recall']:.4f}")
    print(f"confusion   TN={r['tn']}  FP={r['fp']}  FN={r['fn']}  TP={r['tp']}")
    print(f"at natural prevalence 6.64%: precision {r['precision_at_6_64']:.4f}  F1 {r['f1_at_6_64']:.4f}")

p_te = predict_proba(model, X_te)
res_sae = evaluate(y_te, p_te, THRESHOLD)
show(res_sae, "WaferSAE - chronological test window")

--- WaferSAE - chronological test window ---
n=314  defects=17  prevalence=5.41%
PR-AUC (AP) 0.0696  95% CI [0.039, 0.155]   (random = 0.0541)
ROC-AUC     0.5389  95% CI [0.392, 0.687]
F1          0.1364  95% CI [0.047, 0.239]   precision 0.0845  recall 0.3529
confusion   TN=232  FP=65  FN=11  TP=6
at natural prevalence 6.64%: precision 0.1029  F1 0.1593


## 11 · Robustness checks
1. **Seed variance.** Retrain with 5 seeds. Every seed uses the same causal split and its own validation-chosen threshold.
2. **Baseline.** Class-weighted logistic regression on the identical features. Is the deep model earning its complexity?
3. **Walk-forward (expanding-window) backtest.** This simulates monthly retraining in production. For each of 5 consecutive future blocks: refit the preprocessor and model on *all prior* wafers, with the last 20 % of that history as validation, then score the block. Pooled predictions cover far more defects than one test window.

In [12]:
seed_rows = []
for s in range(5):
    m_s, _ = train_sae(X_tr, M_tr, y_tr, X_va, y_va, seed=s, verbose=False)
    thr_s = pick_threshold(y_va, predict_proba(m_s, X_va))
    p_s = predict_proba(m_s, X_te)
    seed_rows.append(dict(seed=s, ap=average_precision_score(y_te, p_s), roc_auc=roc_auc_score(y_te, p_s),
                          f1=f1_score(y_te, p_s >= thr_s, zero_division=0)))
seed_df = pd.DataFrame(seed_rows)
print(seed_df.round(4).to_string(index=False))
print("mean ± std:", {k: f"{seed_df[k].mean():.4f} ± {seed_df[k].std():.4f}" for k in ["ap", "roc_auc", "f1"]})

lr = LogisticRegression(C=0.01, class_weight="balanced", max_iter=5000).fit(X_tr, y_tr)
thr_lr = pick_threshold(y_va, lr.predict_proba(X_va)[:, 1])
res_lr = evaluate(y_te, lr.predict_proba(X_te)[:, 1], thr_lr)
show(res_lr, "Logistic-regression baseline - same test window")

 seed     ap  roc_auc     f1
    0 0.0882   0.5811 0.0000
    1 0.0590   0.4977 0.0000
    2 0.0998   0.6498 0.0000
    3 0.0774   0.5918 0.0000
    4 0.0986   0.6300 0.1579
mean ± std: {'ap': '0.0846 ± 0.0169', 'roc_auc': '0.5901 ± 0.0587', 'f1': '0.0316 ± 0.0706'}


--- Logistic-regression baseline - same test window ---
n=314  defects=17  prevalence=5.41%
PR-AUC (AP) 0.0890  95% CI [0.048, 0.175]   (random = 0.0541)
ROC-AUC     0.6286  95% CI [0.482, 0.767]
F1          0.1412  95% CI [0.075, 0.212]   precision 0.0784  recall 0.7059
confusion   TN=156  FP=141  FN=5  TP=12
at natural prevalence 6.64%: precision 0.0956  F1 0.1684


In [13]:
def walk_forward(n_folds=5, start_frac=0.5):
    edges = np.linspace(int(start_frac * n), n, n_folds + 1).astype(int)
    ys, ps, preds, rows = [], [], [], []
    for k in range(n_folds):
        hist_ix, blk = np.arange(0, edges[k]), np.arange(edges[k], edges[k + 1])
        cut = int(0.8 * len(hist_ix)); tr_ix, va_ix = hist_ix[:cut], hist_ix[cut:]
        pp = CausalPreprocessor().fit(df.iloc[tr_ix])                     # refit on the past only
        (a, am), (b, _), (c, _) = (pp.transform(df.iloc[ix]) for ix in (tr_ix, va_ix, blk))
        m_k, _ = train_sae(a, am, y_all[tr_ix], b, y_all[va_ix], verbose=False)
        thr_k = pick_threshold(y_all[va_ix], predict_proba(m_k, b))
        p_k = predict_proba(m_k, c)
        yk = y_all[blk]
        ys.append(yk); ps.append(p_k); preds.append(p_k >= thr_k)
        rows.append(dict(fold=k, block=f"{df.timestamp[blk[0]]:%m-%d} -> {df.timestamp[blk[-1]]:%m-%d}",
                         n=len(blk), defects=int(yk.sum()),
                         ap=average_precision_score(yk, p_k) if yk.sum() else np.nan,
                         f1=f1_score(yk, p_k >= thr_k, zero_division=0)))
    y, p, pr = np.concatenate(ys), np.concatenate(ps), np.concatenate(preds)
    print(pd.DataFrame(rows).round(4).to_string(index=False))
    tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0, 1]).ravel()
    pooled = dict(n=len(y), defects=int(y.sum()), prevalence=y.mean(),
                  ap=average_precision_score(y, p), roc_auc=roc_auc_score(y, p),
                  f1=f1_score(y, pr), precision=precision_score(y, pr, zero_division=0),
                  recall=recall_score(y, pr), tp=tp, fp=fp, fn=fn, tn=tn)
    print(f"pooled walk-forward: n={pooled['n']} defects={pooled['defects']} prev={pooled['prevalence']:.2%} | "
          f"AP {pooled['ap']:.4f} | ROC-AUC {pooled['roc_auc']:.4f} | F1 {pooled['f1']:.4f} | "
          f"P {pooled['precision']:.4f} R {pooled['recall']:.4f} | TP={tp} FP={fp} FN={fn} TN={tn}")
    return pooled

res_wf = walk_forward()

 fold          block   n  defects     ap     f1
    0 09-11 -> 09-20 156        9 0.0883 0.1791
    1 09-20 -> 09-26 157        2 0.0203 0.0000
    2 09-26 -> 10-02 157        9 0.0531 0.0000
    3 10-02 -> 10-07 157       14 0.1027 0.1026
    4 10-07 -> 10-17 157        3 0.0436 0.0548
pooled walk-forward: n=784 defects=37 prev=4.72% | AP 0.0586 | ROC-AUC 0.5807 | F1 0.0816 | P 0.0481 R 0.2703 | TP=10 FP=198 FN=27 TN=549


## 12 · Release gate (honest)

In [14]:
GATE_F1 = 0.90
passed = res_sae["f1"] >= GATE_F1
print(f"Release gate (F1 ≥ {GATE_F1}) on natural-prevalence test window: {res_sae['f1']:.4f} -> "
      f"{'PASSED' if passed else 'FAILED - NO-GO'}")

os.makedirs("artifacts", exist_ok=True)
torch.save(model.state_dict(), "artifacts/wafer_sae.pt")
joblib.dump(prep, "artifacts/preprocessor.joblib")
summary = dict(threshold=THRESHOLD, features=prep.feature_names_, gate_passed=bool(passed),
               test=dict((k, float(v) if np.isscalar(v) else [float(x) for x in v]) for k, v in res_sae.items()),
               baseline_logreg=dict((k, float(v) if np.isscalar(v) else [float(x) for x in v]) for k, v in res_lr.items()),
               walk_forward=dict((k, float(v)) for k, v in res_wf.items()),
               seeds=seed_df.to_dict(orient="records"))
with open("artifacts/metrics.json", "w") as fh:
    json.dump(summary, fh, indent=2)
print("artifacts written: wafer_sae.pt, preprocessor.joblib, metrics.json "
      "(for traceability only - the gate decision is NO-GO unless PASSED above)")

Release gate (F1 ≥ 0.9) on natural-prevalence test window: 0.1364 -> FAILED - NO-GO
artifacts written: wafer_sae.pt, preprocessor.joblib, metrics.json (for traceability only - the gate decision is NO-GO unless PASSED above)
